In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Libraries imported.")

Libraries imported.


## 1. Versioned output setup

In [ ]:

PROJECT_DIR = Path("/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project")

SOURCE_SPLIT_DIR = PROJECT_DIR / "splits"

DATA_VERSION = "v3_voltage_random10_issue_flight-removed"
OUTPUT_ROOT = PROJECT_DIR / f"model_ready_no_dr_{DATA_VERSION}"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

OVERWRITE = True

print("Project directory:", PROJECT_DIR)
print("Source split directory:", SOURCE_SPLIT_DIR)
print("New output directory:", OUTPUT_ROOT)

Project directory: /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project
Source split directory: /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/splits
New output directory: /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_ready_no_dr_v3_voltage_random10_issue_flight-removed


In [ ]:

# save helpers

def safe_to_parquet(df, path, index=False):
    """
    Parquet is 10-50x faster and uses less disk space.
    The .csv extension in the path is replaced with .parquet automatically.
    """
    path = Path(str(path).replace(".csv", ".parquet"))

    if path.exists() and not OVERWRITE:
        raise FileExistsError(
            f"File already exists and OVERWRITE=False:\n{path}"
        )

    df.to_parquet(path, index=index)
    print("Saved:", path)


def safe_json_dump(obj, path):
    path = Path(path)

    if path.exists() and not OVERWRITE:
        raise FileExistsError(
            f"File already exists and OVERWRITE=False:\n{path}"
        )

    with open(path, "w") as f:
        json.dump(obj, f, indent=4)

    print("Saved:", path)
    
def safe_to_csv(df, path, index=False):
    path = Path(path)

    if path.exists() and not OVERWRITE:
        raise FileExistsError(
            f"File already exists and OVERWRITE=False:\n{path}"
        )

    df.to_csv(path, index=index)
    print("Saved:", path)

## 2. Load existing model-ready selected-feature files

In [ ]:
train_path = SOURCE_SPLIT_DIR / "train_model_ready_selected_features.csv"
val_path   = SOURCE_SPLIT_DIR / "val_model_ready_selected_features.csv"
test_path  = SOURCE_SPLIT_DIR / "test_model_ready_selected_features.csv"

for path in [train_path, val_path, test_path]:
    if not path.exists():
        raise FileNotFoundError(path)

train_base = pd.read_csv(train_path, low_memory=False)
val_base   = pd.read_csv(val_path, low_memory=False)
test_base  = pd.read_csv(test_path, low_memory=False)

print("Original train:", train_base.shape)
print("Original val  :", val_base.shape)
print("Original test :", test_base.shape)

full_df = pd.concat(
    [train_base, val_base, test_base],
    ignore_index=True
)

print("\nCombined full selected-feature dataset:", full_df.shape)
print("Total flights:", full_df["log_id"].nunique())
print(full_df["dataset_label"].value_counts())

Original train: (694549, 19)
Original val  : (138749, 19)
Original test : (138524, 19)

Combined full selected-feature dataset: (971822, 19)
Total flights: 168
dataset_label
normal      690380
abnormal    281442
Name: count, dtype: int64


In [ ]:

# Adding voltage-per-cell features before scaling

def add_battery_voltage_quality_flags(df):
    df = df.copy()

    cell_count_safe = df["battery_status__cell_count"].replace(0, np.nan)

    df["voltage_per_cell"] = (
        df["battery_status__voltage_v"] / cell_count_safe
    )

    df["voltage_filtered_per_cell"] = (
        df["battery_status__voltage_filtered_v"] / cell_count_safe
    )

    # Broad Li-ion/LiPo plausibility range
    # <2.5V per cell = very low / suspicious
    # >4.35V per cell = very high / suspicious
    df["voltage_per_cell_suspicious"] = (
        (df["voltage_per_cell"] < 2.5) |
        (df["voltage_per_cell"] > 4.35) |
        (df["voltage_per_cell"].isna()) |
        (df["voltage_filtered_per_cell"] < 2.5) |
        (df["voltage_filtered_per_cell"] > 4.35) |
        (df["voltage_filtered_per_cell"].isna())
    ).astype(int)

    return df


full_df = add_battery_voltage_quality_flags(full_df)

print("Added new columns:")
print([
    "voltage_per_cell",
    "voltage_filtered_per_cell",
    "voltage_per_cell_suspicious"
])

print("\nSuspicious voltage row counts:")
print(full_df["voltage_per_cell_suspicious"].value_counts())

print("\nSuspicious percentage by label:")
print(
    (full_df.groupby("dataset_label")["voltage_per_cell_suspicious"].mean() * 100)
    .round(2)
)

Added new columns:
['voltage_per_cell', 'voltage_filtered_per_cell', 'voltage_per_cell_suspicious']

Suspicious voltage row counts:
voltage_per_cell_suspicious
0    863609
1    108213
Name: count, dtype: int64

Suspicious percentage by label:
dataset_label
abnormal     4.26
normal      13.94
Name: voltage_per_cell_suspicious, dtype: float64


## 3. Flight removal

In [ ]:


# DATA QUALITY EXCLUSION — f53dabf1
# Justified in notebook 05d_flight_removal_justification

EXCLUDE_FLIGHT_IDS = [
    "f53dabf1-a0fb-433e-8ba8-f09c569db684",
]

flights_before = full_df["log_id"].nunique()
rows_before    = len(full_df)

full_df = full_df[~full_df["log_id"].isin(EXCLUDE_FLIGHT_IDS)].reset_index(drop=True)

flights_after = full_df["log_id"].nunique()
rows_after    = len(full_df)

print("=" * 60)
print("DATA QUALITY EXCLUSION APPLIED")
print("=" * 60)
print(f"Flights before: {flights_before}")
print(f"Flights after:  {flights_after}")
print(f"Flights removed: {flights_before - flights_after}")
print(f"Rows removed:    {rows_before - rows_after:,}")
print(f"\nExcluded flight IDs:")
for fid in EXCLUDE_FLIGHT_IDS:
    print(f"  {fid}")
print(f"\nReason: Sensor logging fault")
print(f"        Cell count = 0 throughout flight")
print(f"        Voltage per cell = 10.27 V (physically impossible)")
print(f"        100% of rows flagged as voltage_per_cell_suspicious")
print(f"        Justified in: 05d_flight_removal_justification.ipynb")


DATA_VERSION = "v4_voltage_random10_cleaned"
print(f"\nData version: {DATA_VERSION}")
print("(v3 results preserved — this creates new output folders)")

DATA QUALITY EXCLUSION APPLIED
Flights before: 168
Flights after:  167
Flights removed: 1
Rows removed:    4,897

Excluded flight IDs:
  f53dabf1-a0fb-433e-8ba8-f09c569db684

Reason: Sensor logging fault
        Cell count = 0 throughout flight
        Voltage per cell = 10.27 V (physically impossible)
        100% of rows flagged as voltage_per_cell_suspicious
        Justified in: 05d_flight_removal_justification.ipynb

Data version: v4_voltage_random10_cleaned
(v3 results preserved — this creates new output folders)


In [ ]:

# to verify if the excluded flight is gone

assert "f53dabf1-a0fb-433e-8ba8-f09c569db684" not in full_df["log_id"].values, \
    "ERROR: Excluded flight still present in full_df"

print("Exclusion verified — faulty flight not in dataset")
print(f"Normal flights:   {(full_df['dataset_label']=='normal').groupby(full_df['log_id']).ngroups}")


Exclusion verified — faulty flight not in dataset
Normal flights:   167


In [ ]:

# Checking one label per flight

label_check = full_df.groupby("log_id")["dataset_label"].nunique()

print("Maximum labels per flight:", label_check.max())

assert label_check.max() == 1, (
    "Some flights have more than one dataset_label. Check before splitting."
)

flight_table = (
    full_df.groupby("log_id")["dataset_label"]
    .first()
    .reset_index()
)

print("\nFlight-level label counts:")
print(flight_table["dataset_label"].value_counts())

Maximum labels per flight: 1

Flight-level label counts:
dataset_label
normal      126
abnormal     41
Name: count, dtype: int64


In [ ]:

COLS_TO_DROP = [
    "battery_status__voltage_cell_v[0]",
    "battery_status__voltage_cell_v[1]",
    "battery_status__voltage_cell_v[2]",
    "battery_status__voltage_cell_v[3]",
    "energy_increment_wh",
]

cols_actually_present = [c for c in COLS_TO_DROP if c in full_df.columns]
cols_not_found = [c for c in COLS_TO_DROP if c not in full_df.columns]

if cols_actually_present:
    full_df = full_df.drop(columns=cols_actually_present)
    print(f"Dropped {len(cols_actually_present)} columns:")
    for c in cols_actually_present:
        print(f"  - {c}")
else:
    print("No columns to drop — already clean.")

if cols_not_found:
    print(f"\nNot found (already removed upstream): {cols_not_found}")

print(f"\nfull_df shape after dropping: {full_df.shape}")

# Verify none of the bad columns remain
for c in COLS_TO_DROP:
    assert c not in full_df.columns, f"Column still present after drop: {c}"

print("Drop check passed.")

# Time-aware repair for voltage-per-cell features

# Logic:
# 1. Mark originally missing voltage-per-cell rows as suspicious.
# 2. Interpolate within each flight only.
# 3. Forward-fill/backward-fill within each flight.
# 4. If still missing, use training-only fallback values.

def repair_voltage_features_before_scaling(train_df, val_df, test_df, split_name):
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    voltage_feature_cols = [
        "voltage_per_cell",
        "voltage_filtered_per_cell"
    ]

    #marking originally missing voltage-per-cell rows

    for df_name, df in [
        ("Train", train_df),
        ("Validation", val_df),
        ("Test", test_df)
    ]:
        missing_rows = df[voltage_feature_cols].isna().any(axis=1)

        if missing_rows.any():
            print(
                f"{split_name} | {df_name}: "
                f"{missing_rows.sum():,} rows have missing voltage-per-cell features. "
                "Marking voltage_per_cell_suspicious = 1."
            )

            df.loc[missing_rows, "voltage_per_cell_suspicious"] = 1

    
    #interpolate within each flight only

    def interpolate_one_df(df, df_name):
        df = df.copy()

        df["_repair_original_order"] = np.arange(len(df))

        df = (
            df.sort_values(["log_id", "time_s"])
            .reset_index(drop=True)
        )

        before_nan = int(df[voltage_feature_cols].isna().sum().sum())

        for col in voltage_feature_cols:
            df[col] = (
                df.groupby("log_id")[col]
                .transform(
                    lambda s: (
                        s.interpolate(method="linear", limit_direction="both")
                         .ffill()
                         .bfill()
                    )
                )
            )

        after_nan = int(df[voltage_feature_cols].isna().sum().sum())

        print(
            f"{split_name} | {df_name}: "
            f"voltage NaNs before interpolation = {before_nan:,}, "
            f"after interpolation = {after_nan:,}"
        )

        df = (
            df.sort_values("_repair_original_order")
            .drop(columns=["_repair_original_order"])
            .reset_index(drop=True)
        )

        return df

    train_df = interpolate_one_df(train_df, "Train")
    val_df = interpolate_one_df(val_df, "Validation")
    test_df = interpolate_one_df(test_df, "Test")

    
    #final fallback from TRAINING data only
    # Used only when a whole flight still has missing voltage values.

    clean_normal_train = train_df[
        (train_df["dataset_label"] == "normal") &
        (train_df["voltage_per_cell_suspicious"] == 0)
    ].copy()

    if len(clean_normal_train) > 0:
        fallback_values = clean_normal_train[voltage_feature_cols].median(numeric_only=True)
        fallback_source = "clean normal training rows"
    else:
        fallback_values = train_df[voltage_feature_cols].median(numeric_only=True)
        fallback_source = "all training rows"

    fallback_values = (
        fallback_values
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    print(f"{split_name}: final fallback calculated from {fallback_source}.")
    print(fallback_values.to_string())

    for df_name, df in [
        ("Train", train_df),
        ("Validation", val_df),
        ("Test", test_df)
    ]:
        before_final = int(df[voltage_feature_cols].isna().sum().sum())

        df[voltage_feature_cols] = (
            df[voltage_feature_cols]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(fallback_values)
        )

        after_final = int(df[voltage_feature_cols].isna().sum().sum())

        print(
            f"{split_name} | {df_name}: "
            f"NaNs before fallback = {before_final:,}, "
            f"after fallback = {after_final:,}"
        )

        assert after_final == 0, (
            f"{split_name} | {df_name}: voltage NaNs still remain."
        )

    return train_df, val_df, test_df


#Scaling setup

target_col = "target_energy_next_60s_wh"

id_cols = [
    "log_id",
    "dataset_label"
]

binary_cols = [
    "voltage_per_cell_suspicious"
]

exclude_from_scaling = id_cols + [target_col] + binary_cols

numeric_cols_to_scale = [
    col for col in full_df.columns
    if col not in exclude_from_scaling
    and pd.api.types.is_numeric_dtype(full_df[col])
]

print(f"\nNumber of continuous numeric columns to scale: {len(numeric_cols_to_scale)}")
print(numeric_cols_to_scale)

# Final check — expect exactly 13 features
assert len(numeric_cols_to_scale) == 13, \
    f"Expected 13 scalable features, got {len(numeric_cols_to_scale)}. Check COLS_TO_DROP."

print("\nBinary columns kept unscaled:")
print(binary_cols)


def scale_train_val_test(train_df, val_df, test_df):
    scaler = StandardScaler()

    train_scaled = train_df.copy()
    val_scaled   = val_df.copy()
    test_scaled  = test_df.copy()

    scaler.fit(train_df[numeric_cols_to_scale])

    train_scaled[numeric_cols_to_scale] = scaler.transform(
        train_df[numeric_cols_to_scale]
    )
    val_scaled[numeric_cols_to_scale] = scaler.transform(
        val_df[numeric_cols_to_scale]
    )
    test_scaled[numeric_cols_to_scale] = scaler.transform(
        test_df[numeric_cols_to_scale]
    )

    return train_scaled, val_scaled, test_scaled, scaler

Dropped 5 columns:
  - battery_status__voltage_cell_v[0]
  - battery_status__voltage_cell_v[1]
  - battery_status__voltage_cell_v[2]
  - battery_status__voltage_cell_v[3]
  - energy_increment_wh

full_df shape after dropping: (966925, 17)
Drop check passed.

Number of continuous numeric columns to scale: 13
['battery_status__cell_count', 'battery_status__current_a', 'battery_status__current_filtered_a', 'battery_status__discharged_mah', 'battery_status__remaining', 'battery_status__scale', 'battery_status__voltage_filtered_v', 'battery_status__voltage_v', 'cumulative_energy_wh', 'power_w', 'time_s', 'voltage_per_cell', 'voltage_filtered_per_cell']

Binary columns kept unscaled:
['voltage_per_cell_suspicious']


In [ ]:
#10 random flight-level splits

SPLIT_SEEDS = [11, 22, 33, 44, 55, 66, 77, 88, 99, 111]

summary_records = []

for split_number, seed in enumerate(SPLIT_SEEDS, start=1):

    split_name = f"split_{split_number:02d}_seed_{seed}"
    split_dir = OUTPUT_ROOT / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "="*80)
    print(split_name)
    print("="*80)

    
    #Flight-level split

    train_flights, temp_flights = train_test_split(
        flight_table,
        test_size=0.30,
        random_state=seed,
        stratify=flight_table["dataset_label"]
    )

    val_flights, test_flights = train_test_split(
        temp_flights,
        test_size=0.50,
        random_state=seed,
        stratify=temp_flights["dataset_label"]
    )

    train_ids = set(train_flights["log_id"])
    val_ids = set(val_flights["log_id"])
    test_ids = set(test_flights["log_id"])

    
    # Create split dataframes

    split_train = full_df[full_df["log_id"].isin(train_ids)].copy()
    split_val = full_df[full_df["log_id"].isin(val_ids)].copy()
    split_test = full_df[full_df["log_id"].isin(test_ids)].copy()

    print("Train:", split_train.shape, "| flights:", split_train["log_id"].nunique())
    print("Val  :", split_val.shape, "| flights:", split_val["log_id"].nunique())
    print("Test :", split_test.shape, "| flights:", split_test["log_id"].nunique())

    print("\nTrain labels:")
    print(split_train["dataset_label"].value_counts())

    print("\nVal labels:")
    print(split_val["dataset_label"].value_counts())

    print("\nTest labels:")
    print(split_test["dataset_label"].value_counts())

   
    # Repair voltage-per-cell features BEFORE scaling

    split_train, split_val, split_test = repair_voltage_features_before_scaling(
        split_train,
        split_val,
        split_test,
        split_name
    )

  
    # Final NaN check before scaling

    for df_name, df in [
        ("Train", split_train),
        ("Validation", split_val),
        ("Test", split_test)
    ]:
        voltage_nan_count = int(
            df[["voltage_per_cell", "voltage_filtered_per_cell"]]
            .isna()
            .sum()
            .sum()
        )

        print(f"{split_name} | {df_name}: voltage NaNs before scaling = {voltage_nan_count}")

        assert voltage_nan_count == 0, (
            f"{split_name} | {df_name}: voltage NaNs found before scaling."
        )

    
    # Scale using train only

    split_train_scaled, split_val_scaled, split_test_scaled, scaler = scale_train_val_test(
        split_train,
        split_val,
        split_test
    )

   
    #Saving unscaled repaired data

    train_unscaled_path = split_dir / f"train_no_dr_interpolated_{DATA_VERSION}_{split_name}.parquet"
    val_unscaled_path   = split_dir / f"val_no_dr_interpolated_{DATA_VERSION}_{split_name}.parquet"
    test_unscaled_path  = split_dir / f"test_no_dr_interpolated_{DATA_VERSION}_{split_name}.parquet"

    safe_to_parquet(split_train, train_unscaled_path)
    safe_to_parquet(split_val, val_unscaled_path)
    safe_to_parquet(split_test, test_unscaled_path)

   
    # Save scaled repaired data
    # These are the files the LSTM notebook will load

    train_scaled_path = split_dir / f"train_no_dr_scaled_{DATA_VERSION}_{split_name}.parquet"
    val_scaled_path   = split_dir / f"val_no_dr_scaled_{DATA_VERSION}_{split_name}.parquet"
    test_scaled_path  = split_dir / f"test_no_dr_scaled_{DATA_VERSION}_{split_name}.parquet"

    safe_to_parquet(split_train_scaled, train_scaled_path)
    safe_to_parquet(split_val_scaled, val_scaled_path)
    safe_to_parquet(split_test_scaled, test_scaled_path)

    #to confirm files were saved

    for check_path in [train_scaled_path, val_scaled_path, test_scaled_path]:
        assert check_path.exists(), f"File was not saved: {check_path}"

    print("Saved all scaled files for:", split_name)

    # Metadata

    metadata = {
        "data_version": DATA_VERSION,
        "split_name": split_name,
        "seed": seed,
        "split_type": "flight_level_random_split",
        "train_ratio": 0.70,
        "validation_ratio": 0.15,
        "test_ratio": 0.15,
        "train_rows": int(len(split_train)),
        "val_rows": int(len(split_val)),
        "test_rows": int(len(split_test)),
        "train_flights": int(split_train["log_id"].nunique()),
        "val_flights": int(split_val["log_id"].nunique()),
        "test_flights": int(split_test["log_id"].nunique()),
        "features_added": [
            "voltage_per_cell",
            "voltage_filtered_per_cell",
            "voltage_per_cell_suspicious"
        ],
        "continuous_scaled_columns": numeric_cols_to_scale,
        "binary_unscaled_columns": binary_cols,
        "target_unscaled": target_col
    }

    metadata_path = split_dir / f"metadata_{DATA_VERSION}_{split_name}.json"
    safe_json_dump(metadata, metadata_path)

    # Add split summary record

    summary_records.append({
        "split_name": split_name,
        "seed": seed,
        "train_rows": len(split_train),
        "val_rows": len(split_val),
        "test_rows": len(split_test),
        "train_flights": split_train["log_id"].nunique(),
        "val_flights": split_val["log_id"].nunique(),
        "test_flights": split_test["log_id"].nunique(),
        "train_abnormal_rows": int((split_train["dataset_label"] == "abnormal").sum()),
        "val_abnormal_rows": int((split_val["dataset_label"] == "abnormal").sum()),
        "test_abnormal_rows": int((split_test["dataset_label"] == "abnormal").sum()),
        "train_suspicious_pct": float(split_train["voltage_per_cell_suspicious"].mean() * 100),
        "val_suspicious_pct": float(split_val["voltage_per_cell_suspicious"].mean() * 100),
        "test_suspicious_pct": float(split_test["voltage_per_cell_suspicious"].mean() * 100),
        "train_scaled_path": str(train_scaled_path),
        "val_scaled_path": str(val_scaled_path),
        "test_scaled_path": str(test_scaled_path)
    })

# Save summary after all 10 splits are created

summary_df = pd.DataFrame(summary_records)

summary_path = OUTPUT_ROOT / f"split_summary_{DATA_VERSION}.csv"
safe_to_csv(summary_df, summary_path)

print("\n" + "="*80)
print("ALL VERSIONED SPLITS CREATED")
print("="*80)

display(summary_df)
print("Summary saved to:", summary_path)

# Final output file check

print("\n" + "="*80)
print("FINAL FILE CHECK")
print("="*80)

for split_number, seed in enumerate(SPLIT_SEEDS, start=1):

    split_name = f"split_{split_number:02d}_seed_{seed}"
    split_dir = OUTPUT_ROOT / split_name

    expected_files = [
        split_dir / f"train_no_dr_scaled_{DATA_VERSION}_{split_name}.parquet",
        split_dir / f"val_no_dr_scaled_{DATA_VERSION}_{split_name}.parquet",
        split_dir / f"test_no_dr_scaled_{DATA_VERSION}_{split_name}.parquet",
    ]

    print(f"\n{split_name}:")

    for file in expected_files:
        print(file.name, "->", file.exists())

        assert file.exists(), f"Missing expected file: {file}"

print("\nAll 10 split folders contain train/val/test scaled parquet files.")


split_01_seed_11
Train: (698993, 17) | flights: 116
Val  : (133277, 17) | flights: 25
Test : (134655, 17) | flights: 26

Train labels:
dataset_label
normal      504192
abnormal    194801
Name: count, dtype: int64

Val labels:
dataset_label
normal      95418
abnormal    37859
Name: count, dtype: int64

Test labels:
dataset_label
normal      90770
abnormal    43885
Name: count, dtype: int64
split_01_seed_11 | Train: voltage NaNs before interpolation = 0, after interpolation = 0
split_01_seed_11 | Validation: voltage NaNs before interpolation = 0, after interpolation = 0
split_01_seed_11 | Test: voltage NaNs before interpolation = 0, after interpolation = 0
split_01_seed_11: final fallback calculated from clean normal training rows.
voltage_per_cell             3.757705
voltage_filtered_per_cell    3.771645
split_01_seed_11 | Train: NaNs before fallback = 0, after fallback = 0
split_01_seed_11 | Validation: NaNs before fallback = 0, after fallback = 0
split_01_seed_11 | Test: NaNs before

,split_name,seed,train_rows,val_rows,test_rows,train_flights,val_flights,test_flights,train_abnormal_rows,val_abnormal_rows,test_abnormal_rows,train_suspicious_pct,val_suspicious_pct,test_suspicious_pct,train_scaled_path,val_scaled_path,test_scaled_path
0,split_01_seed_11,11,698993,133277,134655,116,25,26,194801,37859,43885,10.352607,14.989833,8.149716,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...
1,split_02_seed_22,22,708862,121876,136187,116,25,26,197808,28369,50368,10.427841,9.709869,12.896238,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...
2,split_03_seed_33,33,663320,133584,170021,116,25,26,176028,48849,51668,10.237442,8.419421,14.211186,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...
3,split_04_seed_44,44,677962,129205,159758,116,25,26,183337,49785,43423,11.963797,13.198406,3.225504,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...
4,split_05_seed_55,55,658626,145797,162502,116,25,26,182243,45479,48823,12.773106,8.988525,3.743954,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...
5,split_06_seed_66,66,666087,159038,141800,116,25,26,169436,52626,54483,11.422081,5.176121,13.401269,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...
6,split_07_seed_77,77,686596,129972,150357,116,25,26,188357,32329,55859,13.417060,2.532084,5.256822,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...
7,split_08_seed_88,88,644182,159248,163495,116,25,26,165254,40715,70576,9.804527,13.773485,11.145907,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...
8,split_09_seed_99,99,678359,158054,130512,116,25,26,189506,49199,37840,13.367553,5.183039,3.405051,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...
9,split_10_seed_111,111,685077,129930,151918,116,25,26,198242,36987,41316,12.637995,2.776110,8.642162,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...,/Users/arjun/Documents/UAV thesis/uav-thesis/u...


Summary saved to: /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_ready_no_dr_v3_voltage_random10_issue_flight-removed/split_summary_v4_voltage_random10_cleaned.csv

FINAL FILE CHECK

split_01_seed_11:
train_no_dr_scaled_v4_voltage_random10_cleaned_split_01_seed_11.parquet -> True
val_no_dr_scaled_v4_voltage_random10_cleaned_split_01_seed_11.parquet -> True
test_no_dr_scaled_v4_voltage_random10_cleaned_split_01_seed_11.parquet -> True

split_02_seed_22:
train_no_dr_scaled_v4_voltage_random10_cleaned_split_02_seed_22.parquet -> True
val_no_dr_scaled_v4_voltage_random10_cleaned_split_02_seed_22.parquet -> True
test_no_dr_scaled_v4_voltage_random10_cleaned_split_02_seed_22.parquet -> True

split_03_seed_33:
train_no_dr_scaled_v4_voltage_random10_cleaned_split_03_seed_33.parquet -> True
val_no_dr_scaled_v4_voltage_random10_cleaned_split_03_seed_33.parquet -> True
test_no_dr_scaled_v4_voltage_random10_cleaned_split_03_seed_33.parquet -> True

split_04_seed_44:
tra